# 🔗 LangChain Basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shubh-vedi/awesome-genai-toolkit/blob/main/notebooks/langchain-basics.ipynb)

A comprehensive introduction to LangChain — the most popular framework for building LLM applications.

**What you'll learn:**
- LLM calls and chat models
- Prompt templates
- Output parsers
- Chains (LCEL)
- Memory and conversation
- Simple agent with tools

## 📦 Installation

In [ ]:
!pip install -q langchain langchain-openai langchain-community

## 🔑 Setup

In [ ]:
import os
from google.colab import userdata

# Set your API key (Colab secrets or manual)
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")  # or paste directly

## 1️⃣ Basic LLM Call

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Simple invocation
response = llm.invoke("What is LangChain in one sentence?")
print(response.content)

## 2️⃣ Prompt Templates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a reusable prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {role}. Respond concisely."),
    ("user", "{question}")
])

# Format and invoke
messages = prompt.invoke({"role": "Python developer", "question": "What's the best way to handle errors?"})
response = llm.invoke(messages)
print(response.content)

## 3️⃣ Output Parsers — Get Structured Data

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    rating: float = Field(description="Rating out of 10")
    summary: str = Field(description="One-line summary")

parser = JsonOutputParser(pydantic_object=MovieReview)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a movie critic. {format_instructions}"),
    ("user", "Review the movie: {movie}")
])

chain = prompt | llm | parser
result = chain.invoke({"movie": "Inception", "format_instructions": parser.get_format_instructions()})
print(result)

## 4️⃣ Chains with LCEL (LangChain Expression Language)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Chain: prompt → LLM → parse output
chain = (
    ChatPromptTemplate.from_template("Write a haiku about {topic}")
    | llm
    | StrOutputParser()
)

# Invoke
print(chain.invoke({"topic": "artificial intelligence"}))

# Stream
print("\n--- Streaming ---")
for chunk in chain.stream({"topic": "machine learning"}):
    print(chunk, end="", flush=True)

## 5️⃣ Conversation Memory

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Store for chat history
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("placeholder", "{history}"),
    ("user", "{input}")
])

chain = prompt | llm | StrOutputParser()

with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# First message
config = {"configurable": {"session_id": "demo"}}
print(with_history.invoke({"input": "My name is Alice"}, config=config))

# Follow-up (remembers context)
print(with_history.invoke({"input": "What's my name?"}, config=config))

## 6️⃣ Simple Agent with Tools

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression. Example: '2 + 2' or '15 * 3.5'"""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

@tool
def get_word_count(text: str) -> int:
    """Count the number of words in a text."""
    return len(text.split())

tools = [calculate, get_word_count]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to tools."),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Test the agent
result = executor.invoke({"input": "What is 15% of 2500?"})
print(f"\nAnswer: {result['output']}")

## 🎯 Key Takeaways

| Concept | What It Does |
|---------|-------------|
| `ChatOpenAI` | Connects to OpenAI models |
| `ChatPromptTemplate` | Reusable, parameterized prompts |
| `LCEL (\|)` | Chain components together |
| `OutputParser` | Parse LLM output into structured data |
| `MessageHistory` | Multi-turn conversation memory |
| `@tool` + `AgentExecutor` | Give LLMs the ability to use tools |

## Next Steps
- 📓 [LlamaIndex RAG](llamaindex-rag.ipynb) — Build a full RAG pipeline
- 📓 [CrewAI Agents](crewai-agents.ipynb) — Multi-agent orchestration
- 🔧 [RAG Chatbot App](../apps/rag-chatbot/) — Full app with LangChain